## TP 2 — Profiling & périmètre · lun 25/08 (A. Abarji, journée)
> Objectif : connaître le jeu de données à fond et décider sur quoi porte NutriScope.
1. Profiling systématique des données récupérées au TP 1 : distributions, cardinalités, doublons de codes-barres,
incohérences d'unités, valeurs impossibles (sucres > 100 g/100 g, énergies nulles…).
2. Inventaire des colonnes : lesquelles servent le produit (score, substitution, images, assistant), lesquelles sont du
bruit. S'appuyer sur data-fields.txt d'Open Food Facts.
3. Décision de périmètre en équipe : rayons couverts au lancement (5 à 8 catégories), colonnes conservées, seuil de
complétude minimal par produit.
4. Rédaction de docs/perimetre.md : périmètre retenu, critères, et surtout ce qu'on écarte et pourquoi.
5. Tour des équipes en fin de journée : chaque périmètre est challengé par une autre équipe.
**À committer** : notebook de profiling + docs/perimetre.md argumenté.
> Un périmètre trop large en août se paie en janvier. Le formateur joue le client : il pousse à couper.


##### Récupération des types de données

In [1]:
import pandas as pd
import duckdb
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import plotly.express as px

FOOD_PARQUET = "../data/food.parquet"
FOOD_FR_PARQUET = "../data/food_france.parquet"

In [2]:
# DuckDB : liste des colonnes et de leur type exact
description = duckdb.sql(f"DESCRIBE SELECT * FROM '{FOOD_PARQUET}';").df()
description.to_csv("../data/data_types.csv", index=False, encoding="utf-8")

# Pour une colonne imbriquée en particulier
column_description = duckdb.sql(f"SELECT typeof(nutriments), typeof(packaging) FROM '{FOOD_PARQUET}' LIMIT 1;").df()
for i in range(1) :
    print(column_description.iloc[0,i] + "\n")

STRUCT("name" VARCHAR, "value" FLOAT, "100g" FLOAT, serving FLOAT, unit VARCHAR, prepared_value FLOAT, prepared_100g FLOAT, prepared_serving FLOAT, prepared_unit VARCHAR)[]



### Selection des colonnes

> Voir le fichier [perimetre.md](../docs/perimetre.md)

In [3]:
columns = ", ".join([
    # Général
    "code",
    "product_name",
    "quantity",
    "nutrition_data_per",

    # Classification
    "brands_tags",
    "categories_tags",
    "labels_tags",
    "origins_tags",

    # Ingrédients
    "ingredients_tags",
    "additives_tags",

    "nutriments",
    "nutriscore_grade",
    "nutriscore_score",
    "nutrient_levels_tags",

    "nova_group",

    # Qualité des données
    "completeness",

    # Environnement
    "environmental_score_grade",
    "environmental_score_score",

    # Images
    "images",
])

In [4]:
duckdb.sql(f"""
    COPY (
        SELECT {columns}
        FROM '{FOOD_PARQUET}'
        WHERE list_contains(countries_tags, 'en:france')
    ) TO '{FOOD_FR_PARQUET}' (FORMAT PARQUET)
""")

In [5]:
df = pd.read_parquet(FOOD_FR_PARQUET, engine="pyarrow")

In [6]:
print(df.shape)

(1257105, 19)


#### Distributions

In [7]:
def count_for_graph(df, param):
    return df.groupby(param)[param].count().reset_index(name="somme").sort_values(param)

In [8]:
nutriscore_score = count_for_graph(df,"nutriscore_score")

graph = px.bar(nutriscore_score, x="nutriscore_score", y="somme", color="somme", color_continuous_scale="bluered")
graph.show()

In [9]:
nova_group = count_for_graph(df,"nova_group")

graph = px.bar(nova_group, x="nova_group", y="somme", color="somme", color_continuous_scale="bluered")
graph.show()

In [10]:
environmental_score_score = count_for_graph(df,"environmental_score_score")

graph = px.line(environmental_score_score, x="environmental_score_score", y="somme", title="Nombre de produits par score environnemental")
graph.show()

In [11]:
completeness = count_for_graph(df,"completeness")

graph = px.histogram(completeness, x="completeness", y="somme", nbins=10)
graph.show()

---

## 1. Profiling systématique

Tout le profiling est fait en DuckDB directement sur le Parquet France (`food_france.parquet`,
1 257 105 lignes) : pas de chargement complet en mémoire.


### 1.1 Inventaire des colonnes selectionnées

In [52]:
duckdb.sql(f"""
CREATE OR REPLACE VIEW products AS
SELECT
    code,
    COALESCE(
        list_extract(list_filter(product_name, x -> x.lang = 'fr'),   1)."text",
        list_extract(list_filter(product_name, x -> x.lang = 'main'), 1)."text",
        list_extract(list_filter(product_name, x -> x.lang = 'en'),   1)."text"
    ) AS nom,
    quantity,
    nutrition_data_per,

    brands_tags,
    categories_tags,
    labels_tags,
    origins_tags,

    ingredients_tags,
    additives_tags,

    nutriscore_grade,
    nutriscore_score,

    nova_group,

    completeness,

    environmental_score_grade,
    environmental_score_score,
    
    images,
    
    list_extract(list_filter(nutriments, x -> x.name = 'energy'),        1)."100g" AS energy_100g,
    list_extract(list_filter(nutriments, x -> x.name = 'energy-kcal'),   1)."100g" AS energy_kcal_100g,
    list_extract(list_filter(nutriments, x -> x.name = 'sugars'),        1)."100g" AS sugars_100g,
    list_extract(list_filter(nutriments, x -> x.name = 'salt'),          1)."100g" AS salt_100g,
    list_extract(list_filter(nutriments, x -> x.name = 'fat'),           1)."100g" AS fat_100g,
    list_extract(list_filter(nutriments, x -> x.name = 'saturated-fat'), 1)."100g" AS saturated_fat_100g,
    list_extract(list_filter(nutriments, x -> x.name = 'proteins'),      1)."100g" AS proteins_100g,
    list_extract(list_filter(nutriments, x -> x.name = 'carbohydrates'), 1)."100g" AS carbohydrates_100g
FROM '{FOOD_FR_PARQUET}'
""")

print(duckdb.sql("SELECT COUNT(*) AS produits_france FROM products"))
print(duckdb.sql("select * from products limit 10"))

┌─────────────────┐
│ produits_france │
│      int64      │
├─────────────────┤
│         1257105 │
└─────────────────┘

┌───────────────┬───────────────────────────────────────────────────┬──────────┬────────────────────┬──────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────┬──────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────┬──────────────────┬──────────────────┬────────────┬──────────────┬───────────────────────────┬───────────────────────────┬──────────────────────────────────────────────────────────────────────────────

### 1.2 Doublons et validité des codes-barres

Le code-barres est la clé métier : c'est lui qui portera la contrainte d'unicité dans la base du TP 4.


In [53]:
duckdb.sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(DISTINCT code) AS codes_uniques,
        COUNT(*) - COUNT(DISTINCT code) AS doublons,
    FROM products
""")

┌─────────┬───────────────┬──────────┐
│  total  │ codes_uniques │ doublons │
│  int64  │     int64     │  int64   │
├─────────┼───────────────┼──────────┤
│ 1257105 │       1257078 │       27 │
└─────────┴───────────────┴──────────┘

### 1.3 Taux de remplissage


In [54]:
duckdb.sql("""
    SELECT
        ROUND(100.0 * COUNT(nutriscore_grade) / COUNT(*), 1) AS pct_nutriscore_grade_non_null,
        ROUND(100.0 * COUNT(*) FILTER (nutriscore_grade IN ('a','b','c','d','e')) / COUNT(*), 1) AS pct_nutriscore_exploitable,
        ROUND(100.0 * COUNT(nutriscore_score) / COUNT(*), 1) AS pct_nutriscore_score_non_null,
        ROUND(100.0 * COUNT(nova_group) / COUNT(*), 1) AS pct_nova,
        ROUND(100.0 * COUNT(environmental_score_score) / COUNT(*), 1) AS pct_env_score,
        ROUND(100.0 * COUNT(environmental_score_grade) / COUNT(*), 1) AS pct_env_grade,
        ROUND(100.0 * COUNT(quantity) / COUNT(*), 1) AS pct_quantity
    FROM products
""")

┌───────────────────────────────┬────────────────────────────┬───────────────────────────────┬──────────┬───────────────┬───────────────┬──────────────┐
│ pct_nutriscore_grade_non_null │ pct_nutriscore_exploitable │ pct_nutriscore_score_non_null │ pct_nova │ pct_env_score │ pct_env_grade │ pct_quantity │
│            double             │           double           │            double             │  double  │    double     │    double     │    double    │
├───────────────────────────────┼────────────────────────────┼───────────────────────────────┼──────────┼───────────────┼───────────────┼──────────────┤
│                          98.0 │                       37.1 │                          37.1 │     26.7 │          35.2 │          98.0 │         45.6 │
└───────────────────────────────┴────────────────────────────┴───────────────────────────────┴──────────┴───────────────┴───────────────┴──────────────┘

In [55]:
duckdb.sql("""
    SELECT nutriscore_grade, COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM products GROUP BY 1 ORDER BY n DESC
""")

┌──────────────────┬────────┬────────┐
│ nutriscore_grade │   n    │  pct   │
│     varchar      │ int64  │ double │
├──────────────────┼────────┼────────┤
│ unknown          │ 724267 │   57.6 │
│ e                │ 133097 │   10.6 │
│ d                │ 123443 │    9.8 │
│ c                │  97192 │    7.7 │
│ a                │  63414 │    5.0 │
│ b                │  49615 │    3.9 │
│ not-applicable   │  41306 │    3.3 │
│ NULL             │  24771 │    2.0 │
└──────────────────┴────────┴────────┘

In [61]:
# Colonnes multi-valuées : une liste vide vaut une absence
duckdb.sql("""
    SELECT
        ROUND(100.0 * COUNT(*) FILTER (categories_tags  IS NULL OR len(categories_tags)  = 0) / COUNT(*), 1) AS pct_sans_categorie,
        ROUND(100.0 * COUNT(*) FILTER (brands_tags      IS NULL OR len(brands_tags)      = 0) / COUNT(*), 1) AS pct_sans_marque,
        ROUND(100.0 * COUNT(*) FILTER (ingredients_tags IS NULL OR len(ingredients_tags) = 0) / COUNT(*), 1) AS pct_sans_ingredients,
        ROUND(100.0 * COUNT(*) FILTER (images           IS NULL OR len(images)           = 0) / COUNT(*), 1) AS pct_sans_image
    FROM products
""")

┌────────────────────┬─────────────────┬──────────────────────┬────────────────┐
│ pct_sans_categorie │ pct_sans_marque │ pct_sans_ingredients │ pct_sans_image │
│       double       │     double      │        double        │     double     │
├────────────────────┼─────────────────┼──────────────────────┼────────────────┤
│               50.5 │            48.1 │                 71.4 │            4.6 │
└────────────────────┴─────────────────┴──────────────────────┴────────────────┘

In [64]:
# Taux de manquants sur les nutriments clés
duckdb.sql("""
    SELECT
        ROUND(100.0 * COUNT(*) FILTER (energy_100g IS NULL) / COUNT(*), 1) AS pct_manquant_energy,
        ROUND(100.0 * COUNT(*) FILTER (sugars_100g IS NULL) / COUNT(*), 1) AS pct_manquant_sugars,
        ROUND(100.0 * COUNT(*) FILTER (salt_100g IS NULL) / COUNT(*), 1) AS pct_manquant_salt,
        ROUND(100.0 * COUNT(*) FILTER (fat_100g IS NULL) / COUNT(*), 1) AS pct_manquant_fat,
        ROUND(100.0 * COUNT(*) FILTER (proteins_100g IS NULL) / COUNT(*), 1) AS pct_manquant_proteins
    FROM products
""")

┌─────────────────────┬─────────────────────┬───────────────────┬──────────────────┬───────────────────────┐
│ pct_manquant_energy │ pct_manquant_sugars │ pct_manquant_salt │ pct_manquant_fat │ pct_manquant_proteins │
│       double        │       double        │      double       │      double      │        double         │
├─────────────────────┼─────────────────────┼───────────────────┼──────────────────┼───────────────────────┤
│                28.7 │                29.3 │              33.7 │             29.3 │                  29.1 │
└─────────────────────┴─────────────────────┴───────────────────┴──────────────────┴───────────────────────┘

**Constat.** Le Nutri-Score n'est réellement exploitable que sur **37,1 %** du catalogue France
Les nutriments clés manquent sur ~29 % des produits.
La moitié du catalogue n'a **aucune catégorie**


### 1.4 Cardinalités


In [66]:
duckdb.sql("""
    SELECT
        (SELECT COUNT(DISTINCT b) FROM products, UNNEST(brands_tags)     AS t(b)) AS marques_distinctes,
        (SELECT COUNT(DISTINCT c) FROM products, UNNEST(categories_tags) AS t(c)) AS categories_distinctes,
        (SELECT COUNT(DISTINCT l) FROM products, UNNEST(labels_tags)     AS t(l)) AS labels_distincts,
        (SELECT COUNT(DISTINCT a) FROM products, UNNEST(additives_tags)  AS t(a)) AS additifs_distincts
""")

┌────────────────────┬───────────────────────┬──────────────────┬────────────────────┐
│ marques_distinctes │ categories_distinctes │ labels_distincts │ additifs_distincts │
│       int64        │         int64         │      int64       │       int64        │
├────────────────────┼───────────────────────┼──────────────────┼────────────────────┤
│              90819 │                 38307 │            14028 │                545 │
└────────────────────┴───────────────────────┴──────────────────┴────────────────────┘

In [78]:
top_marques = duckdb.sql("""
    SELECT b AS marque, COUNT(*) AS n
    FROM products, UNNEST(brands_tags) AS t(b)
    GROUP BY 1 ORDER BY n DESC LIMIT 20
""")

px.bar(top_marques.df(), x="n", y="marque", orientation="h",
       title="Les 20 marques les plus présentes sur le périmètre France",
       ).update_yaxes(categoryorder="total ascending").show()

top_marques

┌──────────────────┬───────┐
│      marque      │   n   │
│     varchar      │ int64 │
├──────────────────┼───────┤
│ xx:carrefour     │ 11070 │
│ xx:u             │ 10502 │
│ xx:auchan        │  8298 │
│ xx:marque-repere │  7660 │
│ xx:casino        │  6022 │
│ xx:nestle        │  5972 │
│ xx:leader-price  │  5697 │
│ xx:U             │  5675 │
│ xx:le-gaulois    │  4406 │
│ xx:Carrefour     │  4188 │
│ xx:lidl          │  3993 │
│ xx:monoprix      │  3992 │
│ xx:cora          │  3903 │
│ xx:picard        │  3220 │
│ xx:leclerc       │  2707 │
│ xx:thiriet       │  2623 │
│ xx:danone        │  2501 │
│ xx:la-vie-claire │  2346 │
│ xx:e-leclerc     │  2321 │
│ xx:haribo        │  2066 │
└──────────────────┴───────┘
  20 rows        2 columns

**Constat :** Les `brands_tags` ne sont pas normalisés :
xx:carrefour (11 070) et xx:Carrefour (4 188) coexistent, tout comme xx:u / xx:U
et xx:leclerc / xx:e-leclerc. A normaliser pour que cela soit exploitable


### 1.5 Incohérences d'unités

`nutrition_data_per` indique si les valeurs sont pour 100 g ou pour une portion. Et l'unité
déclarée dans `nutriments` n'est pas toujours celle attendue.


In [80]:
duckdb.sql("""
    SELECT COALESCE(nutrition_data_per, '(null)') AS base_de_reference,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM products GROUP BY 1 ORDER BY n DESC
""")

┌───────────────────┬────────┬────────┐
│ base_de_reference │   n    │  pct   │
│      varchar      │ int64  │ double │
├───────────────────┼────────┼────────┤
│ (null)            │ 906732 │   72.1 │
│ 100g              │ 345544 │   27.5 │
│ serving           │   4812 │    0.4 │
│ 100ml             │     17 │    0.0 │
└───────────────────┴────────┴────────┘

In [81]:
# Combien de produits ont des nutriments alors que la base de référence est inconnue ?
duckdb.sql("""
    SELECT COUNT(*) AS nutriments_sans_base_de_reference
    FROM products
    WHERE nutrition_data_per IS NULL AND energy_100g IS NOT NULL
""")

┌───────────────────────────────────┐
│ nutriments_sans_base_de_reference │
│               int64               │
├───────────────────────────────────┤
│                            826832 │
└───────────────────────────────────┘

**Constat :** 72 % des produits n'indiquent **pas** leur base de référence
(`nutrition_data_per` NULL), dont 826 832 qui ont pourtant des valeurs nutritionnelles :
on ne sait pas si c'est pour 100 g ou pour une portion. 4 812 produits sont
explicitement « par portion » et ne sont donc **pas comparables** aux autres sans conversion.


### 1.6 Valeurs impossibles

Trois familles : bornes physiques (0–100 g pour 100 g), cohérences internes
(sucres ≤ glucides, saturés ≤ matières grasses, somme des macros ≤ 100 g)
et bornes du champ lui-même (`completeness` est censée être entre 0 et 1).


In [97]:
duckdb.sql("""
    SELECT
        COUNT(*) FILTER (sugars_100g   > 100) AS sucres_sup_100g,
        COUNT(*) FILTER (fat_100g      > 100) AS matieres_grasses_sup_100g,
        COUNT(*) FILTER (salt_100g     > 100) AS sel_sup_100g,
        COUNT(*) FILTER (proteins_100g > 100) AS proteines_sup_100g,
        COUNT(*) FILTER (sugars_100g < 0 OR fat_100g < 0 OR salt_100g < 0 OR proteins_100g < 0 OR energy_100g < 0) AS valeurs_negatives,
        COUNT(*) FILTER (energy_100g > 3800) AS energie_sup_3800_kJ
    FROM products
""")

┌─────────────────┬───────────────────────────┬──────────────┬────────────────────┬───────────────────┬─────────────────────┐
│ sucres_sup_100g │ matieres_grasses_sup_100g │ sel_sup_100g │ proteines_sup_100g │ valeurs_negatives │ energie_sup_3800_kJ │
│      int64      │           int64           │    int64     │       int64        │       int64       │        int64        │
├─────────────────┼───────────────────────────┼──────────────┼────────────────────┼───────────────────┼─────────────────────┤
│              49 │                        44 │           55 │                 34 │                 9 │                 121 │
└─────────────────┴───────────────────────────┴──────────────┴────────────────────┴───────────────────┴─────────────────────┘

**Constat** : Dépassement des limites min et max possibles.


### 1.7 Distribution de la complétude


In [91]:
completeness = duckdb.sql("SELECT completeness FROM products WHERE completeness IS NOT NULL").df()

px.histogram(
    completeness,
    x="completeness",
    nbins=40,
    title="La complétude se concentre entre 0,25 et 0,60 — et dépasse 1 pour 11 402 produits"
).show()

---

## 2. Choix des rayons

On cherche des rayons les plus représentés dans la bdd


In [ ]:
top_rayons = duckdb.sql("""
    SELECT b AS rayon, COUNT(*) AS n
    FROM products, UNNEST(categories_tags) AS t(b)
    GROUP BY 1 ORDER BY n DESC LIMIT 20
""")

px.bar(top_rayons.df(), x="n", y="rayon", orientation="h",
       title="Les 20 raysons les plus représentés",
).update_yaxes(categoryorder="total ascending").show()

print(top_rayons)

┌──────────────────────────────────────┬────────┐
│                rayon                 │   n    │
│               varchar                │ int64  │
├──────────────────────────────────────┼────────┤
│ en:plant-based-foods-and-beverages   │ 192272 │
│ en:plant-based-foods                 │ 166862 │
│ en:snacks                            │ 109304 │
│ en:sweet-snacks                      │  91745 │
│ en:meats-and-their-products          │  83912 │
│ en:beverages                         │  71259 │
│ en:meats                             │  63842 │
│ en:dairies                           │  60169 │
│ en:fruits-and-vegetables-based-foods │  51649 │
│ en:cereals-and-potatoes              │  50637 │
│ en:fermented-foods                   │  49056 │
│ en:fermented-milk-products           │  47562 │
│ en:meals                             │  45981 │
│ en:spreads                           │  42738 │
│ en:biscuits-and-cakes                │  40197 │
│ en:desserts                          │  39089 │


Nous avons selectionnés les rayons suivants : 

CATEGORIES = [
    "en:snacks",
    "en:beverages",
    "en:meats",
    "en:fruits-and-vegetables-based-foods",
    "en:cheeses",
    "en:fishes",
    "en:desserts"
]

---

## 3. Conclusion

Le périmètre retenu, ses critères et **ce qu'on écarte et pourquoi** sont argumentés dans
[docs/perimetre.md](../docs/perimetre.md).

Les trois problèmes de qualité :

1. **Marques non normalisées** — casse et alias (`xx:carrefour` / `xx:Carrefour` / `xx:leclerc` / `xx:e-leclerc`).
2. **Base de référence nutritionnelle inconnue** sur 826 832 produits, plus des unités aberrantes
   (kcal dans le champ kJ, sel en mg).
3. **Incohérences internes** des macronutriments (4 029 sommes > 100 g) et `completeness` hors bornes
   (11 402 valeurs > 1).
